# 01 - Dataset EDA and Split Creation

This notebook is intended to run on Kaggle. It downloads the fruit and vegetable healthy/rotten dataset, verifies the 28-class setup, checks class balance, previews sample images, and exports split files for later training notebooks.

Local repository code remains the source of truth. Kaggle is only used as the experiment runner.


## 1. Kaggle setup

If this notebook is opened from inside the cloned GitHub repo on Kaggle, leave `PROJECT_REPO_URL` empty. If you run it as a standalone Kaggle notebook, paste your GitHub repo URL so Kaggle can clone the reusable `src/` code.


In [ ]:
from pathlib import Path
import sys

PROJECT_REPO_URL = "https://github.com/muoimeo/advanced-ai-brfn.git"

current_dir = Path.cwd()

if not (current_dir / "src").exists():
    if PROJECT_REPO_URL:
        repo_name = PROJECT_REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
        if not Path(repo_name).exists():
            !git clone {PROJECT_REPO_URL}
        %cd {repo_name}
        current_dir = Path.cwd()
    else:
        print("src/ was not found. If you are on Kaggle, set PROJECT_REPO_URL and rerun this cell.")

if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

print("Working directory:", Path.cwd())
print("src exists:", (Path.cwd() / "src").exists())


## 2. Imports and configuration


In [ ]:
import json
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from src.config import NUM_CLASSES, RANDOM_SEED, VALID_IMAGE_EXTENSIONS
from src.data.prepare_dataset import build_image_dataframe, stratified_split_dataframe
from src.data.dataloaders import build_class_name_mapping

plt.style.use("default")
pd.set_option("display.max_rows", 40)


## 3. Download dataset

The raw dataset stays in the Kaggle environment. Do not commit it to GitHub.


In [ ]:
DATASET_SLUG = "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten"

download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Downloaded dataset path:", download_path)


## 4. Inspect folder structure

This step is important because some Kaggle datasets have one extra parent folder before the class folders.


In [ ]:
print("Top-level contents:")
for item in sorted(download_path.iterdir()):
    kind = "dir" if item.is_dir() else "file"
    print(f"{kind:4}  {item.name}")


In [ ]:
def count_direct_image_files(directory: Path) -> int:
    count = 0
    for ext in VALID_IMAGE_EXTENSIONS:
        count += len(list(directory.glob(f"*{ext}")))
        count += len(list(directory.glob(f"*{ext.upper()}")))
    return count


def find_dataset_root(start_path: Path, expected_classes: int = NUM_CLASSES) -> Path:
    candidates = []
    for directory in [start_path, *[p for p in start_path.rglob("*") if p.is_dir()]]:
        class_like_dirs = [p for p in directory.iterdir() if p.is_dir() and count_direct_image_files(p) > 0]
        if class_like_dirs:
            candidates.append((directory, len(class_like_dirs)))

    if not candidates:
        raise ValueError(f"Could not find class folders under {start_path}")

    exact_matches = [path for path, class_count in candidates if class_count == expected_classes]
    if exact_matches:
        return exact_matches[0]

    print("No directory with exactly", expected_classes, "class folders was found.")
    print("Candidate dataset roots:")
    for path, class_count in candidates[:10]:
        print(f"- {path} ({class_count} class-like folders)")
    return max(candidates, key=lambda item: item[1])[0]


dataset_root = find_dataset_root(download_path)
print("Selected dataset root:", dataset_root)

class_dirs = sorted([p for p in dataset_root.iterdir() if p.is_dir()])
print("Class folder count:", len(class_dirs))
for class_dir in class_dirs:
    print(class_dir.name)


## 5. Build image dataframe and validate classes


In [ ]:
df = build_image_dataframe(dataset_root)
df["relative_path"] = df["image_path"].apply(lambda p: str(Path(p).relative_to(dataset_root)))

class_names, class_to_index = build_class_name_mapping(df)
df["class_index"] = df["class_name"].map(class_to_index)

print("Total images:", len(df))
print("Number of classes:", len(class_names))
display(df.head())

assert len(class_names) == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, found {len(class_names)}"
assert df["class_name"].isna().sum() == 0
assert df["image_path"].duplicated().sum() == 0


In [ ]:
print("Class names:")
for idx, name in enumerate(class_names):
    print(f"{idx:02d}: {name}")


## 6. Class balance


In [ ]:
counts = df["class_name"].value_counts().sort_index()
summary = counts.rename_axis("class_name").reset_index(name="image_count")
summary["percentage"] = (summary["image_count"] / summary["image_count"].sum() * 100).round(2)
display(summary)

print("Smallest class size:", counts.min())
print("Largest class size:", counts.max())
print("Imbalance ratio:", round(counts.max() / counts.min(), 2))


In [ ]:
fig_height = max(8, len(counts) * 0.32)
plt.figure(figsize=(12, fig_height))
plt.barh(counts.index, counts.values, color="#2f7f6f")
plt.xlabel("Number of images")
plt.ylabel("Class")
plt.title("Class distribution")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 7. Sample images

Use this only for inspection. Training transformations belong in `src/data/augmentations.py` and the training notebooks.


In [ ]:
def show_sample_images(dataframe: pd.DataFrame, samples_per_class: int = 1, max_classes: int = 12) -> None:
    selected_classes = sorted(dataframe["class_name"].unique())[:max_classes]
    sampled = (
        dataframe[dataframe["class_name"].isin(selected_classes)]
        .groupby("class_name", group_keys=False)
        .sample(n=samples_per_class, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    n = len(sampled)
    cols = 4
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 4, rows * 4))

    for i, row in sampled.iterrows():
        image = Image.open(row["image_path"]).convert("RGB")
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(image)
        ax.set_title(row["class_name"], fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_sample_images(df, samples_per_class=1, max_classes=12)


## 8. Train/validation/test split

The split is stratified by the 28 class labels. The exported CSV files are used by later Kaggle training notebooks.


In [ ]:
train_df, val_df, test_df = stratified_split_dataframe(df)

print("Split sizes")
print("train:", len(train_df))
print("val:  ", len(val_df))
print("test: ", len(test_df))

split_counts = pd.DataFrame({
    "train": train_df["class_name"].value_counts().sort_index(),
    "val": val_df["class_name"].value_counts().sort_index(),
    "test": test_df["class_name"].value_counts().sort_index(),
}).fillna(0).astype(int)

split_counts["total"] = split_counts.sum(axis=1)
display(split_counts)

assert len(split_counts) == NUM_CLASSES
assert set(train_df["class_name"]) == set(class_names)
assert set(val_df["class_name"]) == set(class_names)
assert set(test_df["class_name"]) == set(class_names)


## 9. Export EDA artifacts

On Kaggle, these files appear under `/kaggle/working` and can be downloaded after the notebook run. Keep raw images out of GitHub; only bring back lightweight metadata and final model artifacts.


In [ ]:
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/eda")
working_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(working_dir / "train.csv", index=False)
val_df.to_csv(working_dir / "val.csv", index=False)
test_df.to_csv(working_dir / "test.csv", index=False)
summary.to_csv(working_dir / "class_counts.csv", index=False)

with open(working_dir / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

eda_metadata = {
    "dataset_slug": DATASET_SLUG,
    "dataset_root": str(dataset_root),
    "num_images": int(len(df)),
    "num_classes": int(len(class_names)),
    "random_seed": RANDOM_SEED,
    "split_sizes": {
        "train": int(len(train_df)),
        "val": int(len(val_df)),
        "test": int(len(test_df)),
    },
}

with open(working_dir / "eda_metadata.json", "w", encoding="utf-8") as f:
    json.dump(eda_metadata, f, indent=2)

print("Exported files:")
for file_path in sorted(working_dir.glob("*")):
    print(file_path)


## 10. Next step

After this notebook passes, use the exported split files in `02_baseline_cnn.ipynb`. That notebook should import model, dataloader, training, and evaluation helpers from `src/` instead of duplicating core logic.
